### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="houses",
    dataset_year="1990",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://lib.stat.cmu.edu/datasets/",
    download_description="""
We download the houses.zip from the lib.stat.cmu.edu repository and preprocess cadata.txt (strip the 27-line header text, remove leading whitespace, add a column header row).

mkdir -p local-data-warehouse/houses/ && wget -q -P /tmp https://lib.stat.cmu.edu/datasets/houses.zip && unzip -o /tmp/houses.zip -d /tmp && { echo "MedianHouseValue  MedianIncome  HousingMedianAge  TotalRooms  TotalBedrooms  Population  Households  Latitude  Longitude"; sed -n '28,$p' /tmp/cadata.txt | sed 's/^  //'; } > local-data-warehouse/houses/cadata_manual.txt && rm /tmp/houses.zip /tmp/cadata.txt
""",
    # References
    academic_reference_bibtex=r"""@article{pace1997sparse,
  title={Sparse spatial autoregressions},
  author={Pace, R Kelley and Barry, Ronald},
  journal={Statistics \\& Probability Letters},
  volume={33},
  number={3},
  pages={291--297},
  year={1997},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="pace1997sparse",
    license="Public",
    data_tags=["IID", "Spatial"],
    curation_comments="""
- We kept the latitude and longitude features as they are.
- We log scaled the target variable (with base e) as intended by the original task.
- Anomaly: As always, we randomly shuffle the data before uploading. If one does not randomly shuffle the data, there would be a distribution shift from the longitude and latitude based on the original order of data samples.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="LnMedianHouseValue",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/cadata_manual.txt", index_col=False, sep="  ")

# Fix lat/longitude whitespace error from original data
df[["Latitude", "Longitude"]] = df["Latitude"].str.split(" ", expand=True)
df[["Latitude", "Longitude"]] = df[["Latitude", "Longitude"]].astype(float)

target_feature = "LnMedianHouseValue"

# Transform to log space as defined by original task
df[target_feature] = np.log(df["MedianHouseValue"])
df = df.drop(columns=["MedianHouseValue"])

# Spots a distribution shift in the dataset based on the order of the samples, removed by shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

/tmp/ipykernel_288335/1188547135.py:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(f"{dataset_mold.path}/cadata_manual.txt", index_col=False, sep="  ")


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 20,640
Columns: 9
Use sampling: False (sample size: 20,640)
Get row duplicates (staged, merged)...
Using top-8 columns for initial filtering: ['MedianIncome', 'TotalRooms', 'Population', 'TotalBedrooms', 'Households', 'Latitude', 'Longitude', 'HousingMedianAge']
Rows remaining as candidates after top-8 filter: 0 (of 20,640)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,MedianIncome,HousingMedianAge,TotalRooms,TotalBedrooms,Population,Households,Latitude,Longitude,LnMedianHouseValue
0,1.6812,25.0,1505.0,367.0,1392.0,359.0,36.06,-119.01,10.772687
1,2.5313,30.0,2943.0,697.0,1565.0,584.0,35.14,-119.46,10.732039
2,3.4801,52.0,3830.0,1142.0,1310.0,963.0,37.80,-122.44,13.122365
3,5.7376,17.0,3051.0,505.0,1705.0,495.0,34.28,-118.72,12.294999
4,3.7250,34.0,2351.0,440.0,1063.0,428.0,36.62,-121.93,12.535376


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,MedianIncome,float64,0.0,0.0,12928.0,"15.0001, 3.125, 2.875, 4.125, 2.625, 3.875, 3.0, 3.375, 3.625, 4.0"
1,HousingMedianAge,float64,0.0,0.0,52.0,"52.0, 36.0, 35.0, 16.0, 17.0, 34.0, 26.0, 33.0, 18.0, 25.0"
2,TotalRooms,float64,0.0,0.0,5926.0,"1527.0, 1582.0, 1613.0, 2127.0, 1471.0, 1722.0, 1607.0, 2053.0, 1717.0, 1703.0"
3,TotalBedrooms,float64,0.0,0.0,1928.0,"280.0, 331.0, 343.0, 345.0, 393.0, 394.0, 309.0, 328.0, 348.0, 314.0"
4,Population,float64,0.0,0.0,3888.0,"891.0, 850.0, 1227.0, 761.0, 1052.0, 825.0, 999.0, 1005.0, 782.0, 1098.0"
5,Households,float64,0.0,0.0,1815.0,"306.0, 386.0, 335.0, 282.0, 429.0, 375.0, 284.0, 297.0, 340.0, 278.0"
6,Latitude,float64,0.0,0.0,862.0,"34.06, 34.05, 34.08, 34.07, 34.04, 34.09, 34.02, 34.1, 34.03, 33.93"
7,Longitude,float64,0.0,0.0,844.0,"-118.31, -118.3, -118.29, -118.27, -118.32, -118.28, -118.35, -118.36, -118.19, -118.25"
8,LnMedianHouseValue,float64,0.0,0.0,3842.0,"13.1224, 11.8314, 11.9984, 11.6307, 12.1415, 12.3239, 12.7657, 11.3794, 12.5245, 11.9184"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
MedianIncome,20640.0,3.870671,1.899822,0.499900,15.000100
HousingMedianAge,20640.0,28.639486,12.585558,1.000000,52.000000
TotalRooms,20640.0,2635.763081,2181.615252,2.000000,39320.000000
TotalBedrooms,20640.0,537.898014,421.247906,1.000000,6445.000000
Population,20640.0,1425.476744,1132.462122,3.000000,35682.000000
Households,20640.0,499.539680,382.329753,1.000000,6082.000000
Latitude,20640.0,35.631861,2.135952,32.540000,41.950000
Longitude,20640.0,-119.569704,2.003532,-124.350000,-114.310000
LnMedianHouseValue,20640.0,12.084884,0.569134,9.615739,13.122365


In [7]:
# Categorical Feature Statistics
cat_stats

'No categorical/object features to summarize.'

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,-0.173,-0.286,0.324,0.002,log,239353.0,4.278023e+15,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to houses/019d5a72-4a30-70a4-aebc-51ac3e1b5919
019d5a72-4a30-70a4-aebc-51ac3e1b5919
8bf4bbaafd295000a2bddb25a8cd3a52faffc6288b81789e8d360ebedb9a2906
